# Tutorial de ASP y Clingo

Este notebook muestra cómo:
- Cargar Clingo


## 1. Carga y Configuración del Entorno Python

In [1]:
# Instalación del paquete oficial de Clingo para Python
!pip install clingo

In [2]:
import clingo

def resolver_asp(titulo, codigo_asp, opciones=["0"]):
    """
    Función de soporte para instanciar, ejecutar y procesar modelos estables en Clingo.
    - titulos: Cadena de texto para etiquetar y organizar visualmente los resultados
    - codigo_asp: Cadena de texto con el código en ASP escrito en la sintaxis formal de Clingo/Gringo
    - opciones: Lista de parámetros de Clingo (ej. ["0"] indica a Clingo que calcule y devuelva todos los modelos)
    """
    print("=" * 74)
    print(f"  {titulo.upper()}")
    print("=" * 74)

    errores = []

    # Captura los mensajes generados por Clingo
    def logger(codigo, mensaje):
        errores.append(mensaje)

    ctl = clingo.Control(opciones, logger=logger)
    try:
        ctl.add("base", [], codigo_asp)
        ctl.ground([("base", [])])
    except RuntimeError:
        print("\n❌ ERROR EN EL PROGRAMA ASP\n")
        for error in errores:
            print(error)
        return

    modelos = []
    with ctl.solve(yield_=True) as handle:
        for modelo in handle:
            modelos.append(modelo.symbols(shown=True))

    if len(modelos) == 0:
        print("  [ UNSATISFIABLE ] No existe ninguna solución que cumpla las restricciones.\n")
        return

    print(f"\nNúmero de modelos: {len(modelos)}\n")
    for i, modelo in enumerate(modelos, 1):
        print(f"Modelo {i}:")
        print(modelo)

## Caso de estudio 1: sistema autónomo de respuesta ante emergencias con drones de rescate

A través de 4 niveles observaremos cómo la incorporación progresiva de restircciones, tiempo y vairables no monótonas transforma la toma de deciciones del solver.

**Nivel 1: evaluación estática de riesgos (lógica deductiva pura)**

**Objetivo**: iniciar con la deducción de hechos derivados a partir de condiciones estáticas en el mapa.

---

**Qué encontramos**: zonas del mapa, tipos de sensores (humo y calor) y reglas de propagación de riesgo.

**Qué no encontramos**: variables temporales, reglas de elección (*choice rules*), restricciones (`:-`).

**Comportamiento del solver**: produce 1 único *answer set* concluyendo que la zona 2 tiene `riesgo_alto` y la zona 3 `riesgo_medio (tanto por su sensor como por contagio).


In [3]:
asp_nivel_1 = """

% Hechos: Zonas y lectura de sensores
zona(1..4).
alerta_humo(2).
alerta_calor(2).
alerta_humo(3).

zona_colindante(2,3). zona_colindante(3,2).

% Reglas de inferencia deductiva
riesgo_alto(Z) :- alerta_humo(Z), alerta_calor(Z).
riesgo_medio(Z) :- alerta_humo(Z), not riesgo_alto(Z).

% Contagio de riesgo a zonas colindantes
riesgo_medio(Z1) :- riesgo_alto(Z2), zona_colindante(Z1, Z2), not riesgo_alto(Z1).

#show riesgo_alto/1.
#show riesgo_medio/1.
"""

resolver_asp("Nivel 1: emergencias con drones de rescate",asp_nivel_1)

  NIVEL 1: EMERGENCIAS CON DRONES DE RESCATE

Número de modelos: 1

Modelo 1:
[riesgo_alto(2), riesgo_medio(3)]


**Nivel 2: asignación temporal de drones (patrón GDT básico)**

**Objetivo**: introducir la dimensión temporal (T), reglas de elección no deterministas e inercia de extinción.

---

**Qué añadimos respecto al Nivel 1**: el parámetro `#const horizon`, la variable `T`, la asignación de drones a zonas y la prueba de objetivo `(:- incendio(Z, horizon))`.

**Qué quitamos respecto al Nivel 1**: la lógica estática de propagación simple; se pasa a un problema dinámico de respuesta.

**Comportamiento del solver**: se multiplican los modelos estables. Clingo encuentra todas las combinaciones válidas de despliegue donde los drones `d1` y `d2` extinguen los fuegos antes de llegar a `T=2`.

In [29]:
asp_nivel_2 = """

#const horizon = 2.
tiempo(0..horizon).
zona(1..4).
dron(d1; d2).

% Estado inicial: Hay incendios en las zonas 1 y 2
incendio(1, 0).
incendio(2, 0).

% 1. GENERATE: Cada dron puede o no ser asignado a una zona por instante T
{ desplegar(D, Z, T) : zona(Z) } 1 :- dron(D), tiempo(T), T < horizon.

% 2. DEFINE: El fuego se extingue si un dron atiende la zona
extinguido(Z, T+1) :- desplegar(D, Z, T), incendio(Z, T), tiempo(T), T < horizon.
incendio(Z, T+1) :- incendio(Z, T), not extinguido(Z, T+1), tiempo(T), T < horizon.

% 3. TEST: Prohibido terminar el horizonte con incendios activos
:- incendio(Z, horizon).

#show desplegar/3.
"""

resolver_asp("Nivel 2: emergencias con drones de rescate", asp_nivel_2)

  NIVEL 2: EMERGENCIAS CON DRONES DE RESCATE
  Modelo 1: desplegar(d1,1,1) desplegar(d2,2,1)
  Modelo 2: desplegar(d1,3,0) desplegar(d1,1,1) desplegar(d2,2,1)
  Modelo 3: desplegar(d1,4,0) desplegar(d1,1,1) desplegar(d2,2,1)
  Modelo 4: desplegar(d1,1,1) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 5: desplegar(d1,4,0) desplegar(d1,1,1) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 6: desplegar(d1,3,0) desplegar(d1,1,1) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 7: desplegar(d1,1,1) desplegar(d2,4,0) desplegar(d2,2,1)
  Modelo 8: desplegar(d1,3,0) desplegar(d1,1,1) desplegar(d2,4,0) desplegar(d2,2,1)
  Modelo 9: desplegar(d1,4,0) desplegar(d1,1,1) desplegar(d2,4,0) desplegar(d2,2,1)
  Modelo 10: desplegar(d1,2,1) desplegar(d2,1,1)
  Modelo 11: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,1,1)
  Modelo 12: desplegar(d1,2,1) desplegar(d2,4,0) desplegar(d2,1,1)
  Modelo 13: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,4,0) desplegar(d2,1,1)
  Modelo 14: desplegar(d1,4,0) desplega

**Nivel 3: condiciones excepcionales (razonamiento no monótono)**

**Objetivo**: emplear la negación por falla (`not`) para bloquear planes de despliegue cuando surgen interferencias ambientales imprevistas.

---

**Qué añadimos respecto al Nivel 2**: un hecho dinámico de interferencia climática `(fuerte_viento(zona_2, 0))` y una regla de operabilidad condicionada.

**Qué quitamos respecto al Nivel 2**: se mantiene la estructura del Nivel 2, demostrando cómo se corrigen planes sin reescribir la base del código.

**Comportamiento del solver**: el espacio de búsqueda se reduce. Las soluciones que intentaban enviar drones a la zona 2 en `T=0` son descartadas automáticamente por la restricción no monótona. Los drones se ven obligados a esperar a `T=1` para volar a la zona 2.

In [30]:
asp_nivel_3 = """

#const horizon = 2.
tiempo(0..horizon).
zona(1..3).
dron(d1; d2).

incendio(1, 0).
incendio(2, 0).

% HECHO DINÁMICO: Viento extremo en la zona 2 en t=0
fuerte_viento(2, 0).

% Razonamiento No Monótono: Una zona es volable a menos que haya viento fuerte
zona_segura(Z, T) :- zona(Z), tiempo(T), not fuerte_viento(Z, T).

{ desplegar(D, Z, T) : zona(Z) } 1 :- dron(D), tiempo(T), T < horizon.

extinguido(Z, T+1) :- desplegar(D, Z, T), incendio(Z, T), tiempo(T), T < horizon.
incendio(Z, T+1) :- incendio(Z, T), not extinguido(Z, T+1), tiempo(T), T < horizon.

% RESTRICCIÓN ADICIONAL: Prohibido volar a zonas no seguras
:- desplegar(D, Z, T), not zona_segura(Z, T).
:- incendio(Z, horizon).

#show desplegar/3.
"""

resolver_asp("Nivel 3: emergencias con drones de rescate", asp_nivel_3)

  NIVEL 3: EMERGENCIAS CON DRONES DE RESCATE
  Modelo 1: desplegar(d1,1,1) desplegar(d2,2,1)
  Modelo 2: desplegar(d1,3,0) desplegar(d1,1,1) desplegar(d2,2,1)
  Modelo 3: desplegar(d1,1,1) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 4: desplegar(d1,3,0) desplegar(d1,1,1) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 5: desplegar(d1,2,1) desplegar(d2,1,1)
  Modelo 6: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,1,1)
  Modelo 7: desplegar(d1,2,1) desplegar(d2,3,0) desplegar(d2,1,1)
  Modelo 8: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,3,0) desplegar(d2,1,1)
  Modelo 9: desplegar(d1,1,0) desplegar(d2,2,1)
  Modelo 10: desplegar(d1,1,0) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 11: desplegar(d1,1,0) desplegar(d1,3,1) desplegar(d2,2,1)
  Modelo 12: desplegar(d1,1,0) desplegar(d1,3,1) desplegar(d2,3,0) desplegar(d2,2,1)
  Modelo 13: desplegar(d2,1,0) desplegar(d2,2,1)
  Modelo 14: desplegar(d1,3,0) desplegar(d2,1,0) desplegar(d2,2,1)
  Modelo 15: desplegar(d1,3,1) desplegar(d2,

**Nivel 4: gestión de capacidad y batería (agregados cuantitativos)**

**Objetivo**: Emplear agregados numéricos (`#sum` y `#count`) para controlar el gasto de batería de los drones por cada vuelo.

---

**Qué añadimos respecto al Nivel 3**: un consumo de energía por distancia a la zona y una capacidad total de batería por dron.

**Qué quitamos respecto al Nivel 3**: el evento de viento fuerte, para aislar el efecto de la restricción de recursos.

**Comportamiento del solver**: el dron `d2` (batería máx: 4) no puede ser asignado a la zona 2 (coste: 5). Clingo fuerza a que sea exclusivamente el dron `d1` quien asuma la misión lejana a la zona 2.

In [31]:
asp_nivel_4 = """

#const horizon = 2.
tiempo(0..horizon).
zona(1..3).
dron(d1; d2).

% Consumo de batería por viaje según la distancia a la zona
consumo(1, 2). % Zona 1 consume 2 unidades
consumo(2, 5). % Zona 2 consume 5 unidades (muy lejana)
consumo(3, 1).

bateria_maxima(d1, 6).
bateria_maxima(d2, 4).

incendio(1, 0). incendio(2, 0).

{ desplegar(D, Z, T) : zona(Z) } 1 :- dron(D), tiempo(T), T < horizon.

extinguido(Z, T+1) :- desplegar(D, Z, T), incendio(Z, T), tiempo(T), T < horizon.
incendio(Z, T+1) :- incendio(Z, T), not extinguido(Z, T+1), tiempo(T), T < horizon.

% RESTRICCIÓN #SUM: La batería consumida por cada dron no puede exceder su límite
:- dron(D), bateria_maxima(D, Max),
   #sum { C,Z,T : desplegar(D, Z, T), consumo(Z, C) } > Max.

:- incendio(Z, horizon).

#show desplegar/3.
"""

resolver_asp("Nivel 4: emergencias con drones de rescate", asp_nivel_4)

  NIVEL 4: EMERGENCIAS CON DRONES DE RESCATE
  Modelo 1: desplegar(d1,2,1) desplegar(d2,1,1)
  Modelo 2: desplegar(d1,2,1) desplegar(d2,3,0) desplegar(d2,1,1)
  Modelo 3: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,1,1)
  Modelo 4: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,3,0) desplegar(d2,1,1)
  Modelo 5: desplegar(d1,2,0) desplegar(d2,1,1)
  Modelo 6: desplegar(d1,2,0) desplegar(d2,3,0) desplegar(d2,1,1)
  Modelo 7: desplegar(d1,2,0) desplegar(d1,3,1) desplegar(d2,1,1)
  Modelo 8: desplegar(d1,2,0) desplegar(d1,3,1) desplegar(d2,3,0) desplegar(d2,1,1)
  Modelo 9: desplegar(d1,2,1) desplegar(d2,1,0)
  Modelo 10: desplegar(d1,2,1) desplegar(d2,1,0) desplegar(d2,3,1)
  Modelo 11: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,1,0)
  Modelo 12: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,1,0) desplegar(d2,3,1)
  Modelo 13: desplegar(d1,2,1) desplegar(d2,1,0) desplegar(d2,1,1)
  Modelo 14: desplegar(d1,3,0) desplegar(d1,2,1) desplegar(d2,1,0) desplegar(d2,1,1)
  Modelo

## Caso de estudio 2: sistema de asignación inteligente de quirófanos y recursos hospitalarios

A través de 4 niveles, observaremos cómo la programación lógica permite gestionar la asignación de salas, reaccionar ante emergencias médicas imprevistas sin alterar la lógica base y optimizar la carga de trabajo del personal sanitario.

**Nivel 1: compatibilidad quirúrgica (lógica deductiva pura)**

**Objetivo**: deducir qué pacientes pueden intervenirse en qué salas según el equipamiento y la especialidad requerida.

---

**Qué encontramos**: hechos sobre pacientes, quirófanos, especialidades y equipamiento (rayos X, laparoscopia).

**Qué no encontramos**: variable temporal, decisiones voluntarias (*choice rules*), restricciones (`:-`).

**Comportamiento del solver**: genera 1 único *answer set* determinista indicando que `p1` solo es apto en `q2` y `p2` es apto en `q1` y `q2`.

In [32]:
asp_nivel_1 = """

% Hechos: Pacientes y Quirófanos
paciente(p1, cardiologia).
paciente(p2, traumatologia).

quirofano(q1). quirofano(q2).

% Equipamiento disponible
equipamiento(q1, rayos_x).
equipamiento(q2, laparoscopia).
equipamiento(q2, rayos_x).

% Requisitos de la cirugía
requiere(traumatologia, rayos_x).
requiere(cardiologia, laparoscopia).

% Regla deductiva: Un quirófano es apto si tiene todo el equipo requerido
apto(P, Q) :- paciente(P, Esp), quirofano(Q), requiere(Esp, Eq), equipamiento(Q, Eq).

#show apto/2.
"""

resolver_asp("Nivel 1: asignación inteligente de quirófanos y recursos hospitalarios",asp_nivel_1)

  NIVEL 1: ASIGNACIÓN INTELIGENTE DE QUIRÓFANOS Y RECURSOS HOSPITALARIOS
  Modelo 1: apto(p1,q2) apto(p2,q1) apto(p2,q2)

  Total de respuestas válidas: 1



**Nivel 2: programación temporal de cirugías (patrón GDT básico)**

**Objetivo**: asignar horarios de intervención respetando que dos cirugías no coincidan en la misma sala ni en el mismo turno.

---

**Qué añadimos respecto al Nivel 1**: la variable de tiempo `T`, la regla de elección para programar la cirugía y la restricción de solapamiento.

**Qué quitamos respecto al Nivel 1**: simplificamos los requisitos de equipo para aislar la lógica temporal.

**Comportamiento del solver**: genera múltiples modelos estables con todas las permutaciones válidas para intervenir a ambos pacientes en los dos turnos disponibles.

In [33]:
asp_nivel_2 = """

#const horizon = 2.
tiempo(0..horizon).
paciente(p1; p2).
quirofano(q1; q2).

% 1. GENERATE: Asignar a cada paciente exactamente un quirófano y turno T
1 { programar(P, Q, T) : quirofano(Q), tiempo(T) } 1 :- paciente(P).

% 2. DEFINE: Estado de ocupación de la sala
ocupado(Q, T) :- programar(_, Q, T).

% 3. TEST: Prohibido programar dos pacientes en la misma sala al mismo tiempo
:- programar(P1, Q, T), programar(P2, Q, T), P1 != P2.

#show programar/3.
"""

resolver_asp("Nivel 2: asignación inteligente de quirófanos y recursos hospitalarios",asp_nivel_2)

  NIVEL 2: ASIGNACIÓN INTELIGENTE DE QUIRÓFANOS Y RECURSOS HOSPITALARIOS
  Modelo 1: programar(p2,q1,2) programar(p1,q2,0)
  Modelo 2: programar(p1,q1,1) programar(p2,q1,2)
  Modelo 3: programar(p1,q2,0) programar(p2,q2,2)
  Modelo 4: programar(p1,q1,1) programar(p2,q2,2)
  Modelo 5: programar(p2,q1,2) programar(p1,q2,2)
  Modelo 6: programar(p1,q2,0) programar(p2,q2,1)
  Modelo 7: programar(p1,q2,2) programar(p2,q2,1)
  Modelo 8: programar(p1,q1,1) programar(p2,q2,1)
  Modelo 9: programar(p1,q1,2) programar(p2,q2,2)
  Modelo 10: programar(p1,q1,2) programar(p2,q2,1)
  Modelo 11: programar(p2,q1,2) programar(p1,q2,1)
  Modelo 12: programar(p1,q2,1) programar(p2,q2,2)
  Modelo 13: programar(p1,q2,1) programar(p2,q2,0)
  Modelo 14: programar(p1,q2,2) programar(p2,q2,0)
  Modelo 15: programar(p1,q1,1) programar(p2,q2,0)
  Modelo 16: programar(p1,q1,2) programar(p2,q2,0)
  Modelo 17: programar(p2,q1,1) programar(p1,q2,0)
  Modelo 18: programar(p2,q1,1) programar(p1,q2,1)
  Modelo 19: progr

**Nivel 3: reprogramación por urgencia crítica (razonamiento no monótono)**

**Objetivo**: usar la negación por falla (`not`) para desalojar o cancelar una cirugía electiva si se presenta un paciente con prioridad vital.

---

**Qué añadimos respecto al Nivel 2**: la llegada imprevista de un paciente crítico `(urgencia_vital(p_critico, 0))` que requiere bloqueo de quirófano.

**Qué quitamos respecto al Nivel 2**: ningún elemento del código base; se muestra la inercia y la cancelación automática mediante `not`.

**Comportamiento del solver**: las opciones que intentaban asignar a los pacientes `p1` o `p2` al quirófano `q1` en `T=0` colapsan. El solver desplaza automáticamente las cirugías ordinarias a turnos posteriores.

In [34]:
asp_nivel_3 = """

#const horizon = 2.
tiempo(0..horizon).
paciente(p1; p2).
quirofano(q1).

% HECHO DINÁMICO: Llega una urgencia médica en t=0 que requiere el quirófano q1
urgencia_vital(p_critico, q1, 0).

% Razonamiento No Monótono: Una sala está disponible para cirugías programadas A MENOS QUE esté reservada por urgencia
sala_disponible(Q, T) :- quirofano(Q), tiempo(T), not urgencia_vital(_, Q, T).

% GENERATE: Solo se puede programar en salas disponibles
1 { programar(P, Q, T) : sala_disponible(Q, T), tiempo(T) } 1 :- paciente(P).

:- programar(P1, Q, T), programar(P2, Q, T), P1 != P2.

#show programar/3.
#show urgencia_vital/3.
"""

resolver_asp("Nivel 3: asignación inteligente de quirófanos y recursos hospitalarios",asp_nivel_3)

  NIVEL 3: ASIGNACIÓN INTELIGENTE DE QUIRÓFANOS Y RECURSOS HOSPITALARIOS
  Modelo 1: programar(p2,q1,1) programar(p1,q1,2) urgencia_vital(p_critico,q1,0)
  Modelo 2: programar(p1,q1,1) programar(p2,q1,2) urgencia_vital(p_critico,q1,0)

  Total de respuestas válidas: 2



**Nivel 4: control de carga de trabajo y fatiga (agregados cuantitativos)**

**Objetivo**: emplear agregados numéricos (`#sum`) para limitar las horas continuas de trabajo asignadas a un mismo cirujano.

---

**Qué añadimos respecto al Nivel 3**: horas de duración por cirugía y un límite máximo de horas por cirujano.

**Qué quitamos respecto al Nivel 3**: la urgencia vital para centrar la atención en el filtrado cuantitativo.

**Comportamiento del solver**: descarta combinaciones donde un solo cirujano asume a `p1` y `p2` (suma 2+2=4>3). Fuerza la distribución del trabajo entre `c1` y `c2`.

In [35]:
asp_nivel_4 = """

#const horizon = 3.
tiempo(0..horizon).
paciente(p1; p2; p3).
cirujano(c1; c2).

duracion(p1, 2). % p1 requiere 2 horas
duracion(p2, 2). % p2 requiere 2 horas
duracion(p3, 1). % p3 requiere 1 hora

#const max_horas = 3.

% GENERATE: Asignar cirujano a cada paciente
1 { asignar_cirujano(P, C) : cirujano(C) } 1 :- paciente(P).

% TEST CON AGREGADO #SUM: Ningún cirujano puede superar las horas máximas asignadas
:- cirujano(C), #sum { H,P : asignar_cirujano(P, C), duracion(P, H) } > max_horas.

#show asignar_cirujano/2.
"""

resolver_asp("Nivel 4: asignación inteligente de quirófanos y recursos hospitalarios",asp_nivel_4)

  NIVEL 4: ASIGNACIÓN INTELIGENTE DE QUIRÓFANOS Y RECURSOS HOSPITALARIOS
  Modelo 1: asignar_cirujano(p2,c1) asignar_cirujano(p3,c1) asignar_cirujano(p1,c2)
  Modelo 2: asignar_cirujano(p2,c1) asignar_cirujano(p1,c2) asignar_cirujano(p3,c2)
  Modelo 3: asignar_cirujano(p1,c1) asignar_cirujano(p2,c2) asignar_cirujano(p3,c2)
  Modelo 4: asignar_cirujano(p1,c1) asignar_cirujano(p3,c1) asignar_cirujano(p2,c2)

  Total de respuestas válidas: 4



## Caso de estudio 3: sistema autónomo de ciberseguridad y respuesta a incidentes en una red de servidores

A través de 4 niveles, aprenderá a modelar la propagación de *malware* en una red, aislar nodos comprometidos mediante decisiones temporales, adaptar la respuesta ante vulnerabilidades *zero-day* (o de día cero) usando razonamiento no monótono y optimizar el impacto en el negocio.

**Nivel 1: propagación de amenazas (lógica deductiva pura)**

**Objetivo**: comprender la deducción recursiva en grafos de red sin dimensión temporal ni intervención.

---

**Qué encontramos**: servidores, conexiones de red y reglas de propagación determinista de *malware*.

**Qué no encontramos**: variable temporal, decisiones voluntarias (*choice rules*), restricciones (`:-`).

**Comportamiento del solver**: genera 1 único *answer set* determinista mostrando que, sin defensas, toda la red (`web`, `app`, `db`, `backup`) queda comprometida.

In [36]:
asp_nivel_1 = """

% Hechos: Topología de red y servidor inicialmente infectado
servidor(web; app; db; backup).
conexion(web, app). conexion(app, db). conexion(app, backup).

infectado(web).

% Regla deductiva: El malware se propaga recursivamente por las conexiones
comprometido(S) :- infectado(S).
comprometido(Y) :- comprometido(X), conexion(X, Y).

#show comprometido/1.
"""

resolver_asp("Nivel 1: sistema autónomo de ciberseguridad",asp_nivel_1)

  NIVEL 1: SISTEMA AUTÓNOMO DE CIBERSEGURIDAD
  Modelo 1: comprometido(web) comprometido(app) comprometido(db) comprometido(backup)

  Total de respuestas válidas: 1



**Nivel 2: contención temporal de incidentes (patrón GDT básico)**

**Objetivo**: introducir la dimensión temporal (T), el aislamiento dinámico de nodos y la prueba de seguridad en la base de datos crítica.

---

**Qué añadimos respecto al Nivel 1**: parámetro `#const horizon`, tiempo `T`, la acción de aislar servidores y la restricción de que la base de datos (`db`) debe permanecer a salvo en el horizonte final.

**Qué quitamos respecto al Nivel 1**: eliminamos la propagación estática infinita; la infección avanza paso a paso en el tiempo.

**Comportamiento del solver**: genera las secuencias de aislamiento válidas (por ejemplo, aislar `app` en `T=0` o en `T=1`) para impedir que la infección alcance db.

In [37]:
asp_nivel_2 = """

#const horizon = 2.
tiempo(0..horizon).
servidor(web; app; db; backup).
conexion(web, app). conexion(app, db). conexion(app, backup).

infectado(web, 0).

% 1. GENERATE: El SOC puede aislar como máximo 1 servidor por paso de tiempo
{ aislar(S, T) : servidor(S) } 1 :- tiempo(T), T < horizon.

% 2. DEFINE: Estado de aislamiento y propagación temporal
aislado(S, T+1) :- aislar(S, T), tiempo(T), T < horizon.
aislado(S, T+1) :- aislado(S, T), tiempo(T), T < horizon. % Inercia del aislamiento

% El malware avanza solo a nodos que NO estén aislados
infectado(Y, T+1) :- infectado(X, T), conexion(X, Y), not aislado(Y, T), not aislado(X, T), tiempo(T), T < horizon.
infectado(S, T+1) :- infectado(S, T), tiempo(T), T < horizon. % Inercia de infección

% 3. TEST: Prohibido que la base de datos se contamine al final del horizonte
:- infectado(db, horizon).

#show aislar/2.
#show infectado/2.
"""

resolver_asp("Nivel 2: sistema autónomo de ciberseguridad",asp_nivel_2)

  NIVEL 2: SISTEMA AUTÓNOMO DE CIBERSEGURIDAD
  Modelo 1: aislar(db,0) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2) infectado(backup,2)
  Modelo 2: aislar(db,0) aislar(db,1) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2) infectado(backup,2)
  Modelo 3: aislar(db,0) aislar(app,1) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2) infectado(backup,2)
  Modelo 4: aislar(db,0) aislar(web,1) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2) infectado(backup,2)
  Modelo 5: aislar(db,0) aislar(backup,1) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2) infectado(backup,2)
  Modelo 6: aislar(app,0) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2)
  Modelo 7: aislar(app,0) aislar(app,1) infectado(web,0) infectado(web,1) infectado(app,1) infectado(app,2) infectado(web,2)
  Modelo 8

**Nivel 3: vulnerabilidades *zero-day* e inercia (razonamiento no monótono)**

**Objetivo**: usar la negación por falla (`not`) para retirar la confianza en una ruta de red cuando se descubre un exploit Zero-Day de forma imprevista.

---

**Qué añadimos respecto al Nivel 2**: un hecho dinámico de exploit descubierto `(zero_day(app, 0))` y una regla de canal seguro no monótona.

**Qué quitamos respecto al Nivel 2**: se mantiene la estructura del nivel 2 para ver cómo la base de reglas reacciona adaptativamente a la nueva información.

**Comportamiento del solver**: al declararse la presencia del `zero_day`, el solver entiende que la infección se propagará más rápido por esa vía y reordena las prioridades de aislamiento para proteger la base de datos a tiempo.

In [38]:
asp_nivel_3 = """

#const horizon = 2.
tiempo(0..horizon).
servidor(web; app; db).
conexion(web, app). conexion(app, db).

infectado(web, 0).

% HECHO DINÁMICO: Se detecta una vulnerabilidad crítica Zero-Day en 'app' en t=0
zero_day(app, 0).

% Razonamiento No Monótono: Una conexión es confiable A MENOS QUE se demuestre un Zero-Day
conexion_segura(X, Y, T) :- conexion(X, Y), tiempo(T), not zero_day(X, T), not zero_day(Y, T).

{ aislar(S, T) : servidor(S) } 1 :- tiempo(T), T < horizon.
aislado(S, T+1) :- aislar(S, T), tiempo(T), T < horizon.

% Infección acelerada por canal no seguro
infectado(Y, T+1) :- infectado(X, T), conexion(X, Y), not conexion_segura(X, Y, T), tiempo(T), T < horizon.
infectado(S, T+1) :- infectado(S, T), tiempo(T), T < horizon.

:- infectado(db, horizon).

#show aislar/2.
#show zero_day/2.
"""

resolver_asp("Nivel 3: sistema autónomo de ciberseguridad",asp_nivel_3)

  NIVEL 3: SISTEMA AUTÓNOMO DE CIBERSEGURIDAD
  Modelo 1: zero_day(app,0)
  Modelo 2: zero_day(app,0) aislar(db,0)
  Modelo 3: zero_day(app,0) aislar(web,1)
  Modelo 4: zero_day(app,0) aislar(db,0) aislar(web,1)
  Modelo 5: zero_day(app,0) aislar(app,1)
  Modelo 6: zero_day(app,0) aislar(db,0) aislar(app,1)
  Modelo 7: zero_day(app,0) aislar(db,1)
  Modelo 8: zero_day(app,0) aislar(db,0) aislar(db,1)
  Modelo 9: zero_day(app,0) aislar(app,0)
  Modelo 10: zero_day(app,0) aislar(app,0) aislar(web,1)
  Modelo 11: zero_day(app,0) aislar(app,0) aislar(app,1)
  Modelo 12: zero_day(app,0) aislar(app,0) aislar(db,1)
  Modelo 13: zero_day(app,0) aislar(web,0)
  Modelo 14: zero_day(app,0) aislar(web,0) aislar(app,1)
  Modelo 15: zero_day(app,0) aislar(web,0) aislar(db,1)
  Modelo 16: zero_day(app,0) aislar(web,0) aislar(web,1)

  Total de respuestas válidas: 16



**Nivel 4: capacidad de cortafuegos y ancho de banda (agregados cuantitativos)**

**Objetivo**: emplear agregados numéricos (`#sum`) para limitar el consumo de CPU o capacidad del cortafuegos al aplicar reglas de aislamiento.

---

**Qué añadimos respecto al Nivel 3**: coste computacional por aislar cada tipo de servidor y una capacidad máxima del *firewall* por turno.

**Qué quitamos respecto al Nivel 3**: el evento *zero-day* para aislar el estudio de la restricción de recursos.

**Comportamiento del solver**: el aislamiento de `app` (coste: 5) es descartado por exceder la capacidad de CPU del cortafuegos (máx: 4). Clingo busca estrategias alternativas (como aislar `web` y `backup` por separado) que se ajusten al presupuesto de recursos.

In [39]:
asp_nivel_4 = """

#const horizon = 2.
tiempo(0..horizon).
servidor(web; app; db; backup).
conexion(web, app). conexion(app, db). conexion(app, backup).

% Coste de procesamiento de cortafuegos según la complejidad del servidor
coste_aislamiento(web, 2).
coste_aislamiento(app, 5). % Alto coste por ser el núcleo de servicios
coste_aislamiento(backup, 1).

#const cpu_firewall_max = 4.

infectado(web, 0).

{ aislar(S, T) : servidor(S) } :- tiempo(T), T < horizon.
aislado(S, T+1) :- aislar(S, T), tiempo(T), T < horizon.
aislado(S, T+1) :- aislado(S, T), tiempo(T), T < horizon.

infectado(Y, T+1) :- infectado(X, T), conexion(X, Y), not aislado(Y, T), not aislado(X, T), tiempo(T), T < horizon.
infectado(S, T+1) :- infectado(S, T), tiempo(T), T < horizon.

% RESTRICCIÓN #SUM: El coste total de aislamiento en cada turno T no puede superar la CPU del firewall
:- tiempo(T), #sum { C,S : aislar(S, T), coste_aislamiento(S, C) } > cpu_firewall_max.

:- infectado(db, horizon).

#show aislar/2.
"""
resolver_asp("Nivel 4: sistema autónomo de ciberseguridad",asp_nivel_4)


  NIVEL 4: SISTEMA AUTÓNOMO DE CIBERSEGURIDAD
  Modelo 1: aislar(db,0)
  Modelo 2: aislar(db,0) aislar(db,1)
  Modelo 3: aislar(db,0) aislar(backup,1)
  Modelo 4: aislar(db,0) aislar(db,1) aislar(backup,1)
  Modelo 5: aislar(db,0) aislar(web,1)
  Modelo 6: aislar(db,0) aislar(web,1) aislar(db,1)
  Modelo 7: aislar(web,0) aislar(db,0)
  Modelo 8: aislar(web,0) aislar(db,0) aislar(db,1)
  Modelo 9: aislar(web,0) aislar(db,0) aislar(web,1)
  Modelo 10: aislar(web,0) aislar(db,0) aislar(web,1) aislar(db,1)
  Modelo 11: aislar(web,0) aislar(db,0) aislar(backup,1)
  Modelo 12: aislar(web,0) aislar(db,0) aislar(db,1) aislar(backup,1)
  Modelo 13: aislar(db,0) aislar(web,1) aislar(backup,1)
  Modelo 14: aislar(db,0) aislar(web,1) aislar(db,1) aislar(backup,1)
  Modelo 15: aislar(web,0) aislar(db,0) aislar(web,1) aislar(backup,1)
  Modelo 16: aislar(web,0) aislar(db,0) aislar(web,1) aislar(db,1) aislar(backup,1)
  Modelo 17: aislar(db,0) aislar(backup,0)
  Modelo 18: aislar(db,0) aislar(backup,